In [32]:
import pandas as pd
import os
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive

# Isso vai abrir uma janelinha pedindo autorização para acessar seu Drive
drive.mount('/content/drive')

In [33]:
from google.colab import files
import io

uploaded = files.upload()

nome_arquivo_local = list(uploaded.keys())[0]
df_local = pd.read_excel(io.BytesIO(uploaded[nome_arquivo_local]))

print(f"Arquivo {nome_arquivo_local} carregado com sucesso do seu PC!")
df_local.head(20)

Saving Pesquisa_Mercado_Livre_2026-03-17_08-27-04.xlsx to Pesquisa_Mercado_Livre_2026-03-17_08-27-04 (1).xlsx
Arquivo Pesquisa_Mercado_Livre_2026-03-17_08-27-04 (1).xlsx carregado com sucesso do seu PC!


,Produto,Preço (R$),Frete (R$),Selo_Full,Link
0,Sony PlayStation CUH-2215A 4 Slim 500gb Standa...,1983,0,False,https://www.mercadolivre.com.br/sony-playstati...
1,Sony PlayStation 4 Slim 500GB,1999,0,False,https://www.mercadolivre.com.br/sony-playstati...
2,Sony PlayStation 4 Slim 1TB Standard Preto Onyx,2019,0,False,https://www.mercadolivre.com.br/sony-playstati...
3,Sony Playstation 4 Ps4 Slim 1tb Standard Preto...,2498,0,True,https://www.mercadolivre.com.br/sony-playstati...
4,PlayStation 5 Slim Cor Branco 1 TB Versão Mídi...,3812,0,True,https://click1.mercadolivre.com.br/mclics/clic...
5,Console Playstation 5 Slim Edição Digital 825 Gb,3419,0,True,https://click1.mercadolivre.com.br/mclics/clic...
6,Máquinas Portáteis De Videogame Para Adultos E...,201,0,False,https://click1.mercadolivre.com.br/mclics/clic...
7,Rox Portátil Digital Playstation R36 Reproduct...,274,0,False,https://click1.mercadolivre.com.br/mclics/clic...
8,"Sony PlayStation 4 Slim 1TB DualShock 4, 2 Con...",2196,0,False,https://www.mercadolivre.com.br/sony-playstati...
9,Sony PlayStation 4 Slim 1TB FIFA 19 Bundle cor...,2169,0,False,https://www.mercadolivre.com.br/sony-playstati...


In [38]:
# Criando a coluna de Custo Total
df_local['Custo_Total'] = df_local['Preço (R$)'] + df_local['Frete (R$)']

# Ordenando do mais barato para o mais caro
df_local = df_local.sort_values(by='Custo_Total').reset_index(drop=True)

print("Cálculos concluídos! Aqui estão as 5 ofertas mais baratas (Custo Total):")
df_local[['Produto', 'Custo_Total', 'Selo_Full']].head(5)

Cálculos concluídos! Aqui estão as 5 ofertas mais baratas (Custo Total):


,Produto,Custo_Total,Selo_Full
0,Máquinas Portáteis De Videogame Para Adultos E...,201,False
1,Rox Portátil Digital Playstation R36 Reproduct...,274,False
2,Sony PlayStation 4 Slim 500GB Uncharted 4: A T...,1599,False
3,Console Sony Playstation 4 Slim 500gb Original...,1600,False
4,Sony PlayStation 4 Slim 1TB Standard Preto Ony...,1635,False


In [39]:
# Filtrando apenas os que são FULL e pegando o "campeão" de preço
# Nota: Garantimos que o Selo_Full seja tratado como booleano (True/False)
melhores_ofertas_full = df_local[df_local['Selo_Full'] == True].head(3)

if not melhores_ofertas_full.empty:
    vencedor = melhores_ofertas_full.iloc[0]
    print(f"🏆 RECOMENDAÇÃO DA IA: \nO produto '{vencedor['Produto']}' é a melhor escolha.")
    print(f"Custo: R$ {vencedor['Custo_Total']:.2f} com entrega FULL.")
else:
    print("Nenhuma oferta com selo FULL encontrada para recomendação automática.")

🏆 RECOMENDAÇÃO DA IA: 
O produto '# Ps4 Play Slim 1 Tb Tera Sony Playstation Garantia Nf Top Preto (Recondicionado)' é a melhor escolha.
Custo: R$ 1842.00 com entrega FULL.


In [56]:
import pandas as pd
import plotly.express as px
from IPython.display import HTML, display

# 1. Preparação dos Dados e Limpeza de Duplicados para o Gráfico
df_local['Custo_Total'] = df_local['Preço (R$)'] + df_local['Frete (R$)']
df_local = df_local.sort_values(by='Custo_Total', ascending=True).reset_index(drop=True)

# Pegamos os top 30 e garantimos que não haja nomes duplicados para não bugar as barras
df_30 = df_local.head(30).copy()
df_30['Nome_Curto'] = df_30['Produto'].str[:35] + "..."
df_plot = df_30.drop_duplicates(subset=['Nome_Curto']).copy()

# 2. Lógica de Cores para o Ranking
cores_ranking = []
for i, row in df_plot.iterrows():
    if i == 0: cores_ranking.append('#FFD700') # Ouro
    elif i == 1: cores_ranking.append('#C0C0C0') # Prata
    elif i == 2: cores_ranking.append('#CD7F32') # Bronze
    else: cores_ranking.append('#10B981' if row['Selo_Full'] else '#475569')

# 3. GRÁFICO Esquerda: Ranking Horizontal (Sem Barras Duplas)
fig_bar = px.bar(df_plot[::-1],
                 x="Custo_Total", y="Nome_Curto",
                 orientation='h', template="plotly_dark",
                 hover_data={'Link': False, 'Custo_Total': ':.2f'})

fig_bar.update_traces(
    marker_color=cores_ranking[::-1],
    texttemplate='R$ %{x:.2f}', textposition='outside',
    customdata=df_plot['Link'][::-1],
    cliponaxis=False # Garante que o texto do preço não seja cortado
)

fig_bar.update_layout(
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(showticklabels=False, showgrid=False, range=[0, df_plot['Custo_Total'].max()*1.3]),
    yaxis=dict(title="", tickfont=dict(color="#ffffff", size=10)),
    margin=dict(l=0, r=80, t=0, b=0), height=700,
    showlegend=False
)

# 4. GRÁFICO Direita: Pizza Breakdown
fig_pie = px.pie(df_local, names='Entrega Full ⚡', hole=0.6,
                 color='Entrega Full ⚡',
                 color_discrete_map={'Sim': '#10B981', 'Não': '#475569'},
                 template="plotly_dark")
fig_pie.update_layout(showlegend=False, margin=dict(t=0, b=0, l=0, r=0), height=230)
fig_pie.update_traces(textinfo='percent', textfont_size=14)

# 5. Montagem do HTML Final
id_div = "market_dash_final_v3"
graph_bar_html = fig_bar.to_html(full_html=False, include_plotlyjs='cdn', div_id=id_div)
graph_pie_html = fig_pie.to_html(full_html=False, include_plotlyjs='cdn')

dashboard_html = f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;700&display=swap');
    .main-box {{ background: #020617; font-family: 'Inter', sans-serif; padding: 35px; border-radius: 20px; color: white; border: 1px solid #1e293b; max-width: 1150px; }}
    .kpi-row {{ display: flex; gap: 15px; margin-bottom: 25px; }}
    .kpi-card {{ flex: 1; background: #0f172a; padding: 15px 20px; border-radius: 12px; border: 1px solid #1e293b; }}
    .kpi-card span {{ color: #94a3b8; font-size: 10px; font-weight: bold; text-transform: uppercase; letter-spacing: 1px; }}
    .kpi-card h2 {{ color: #ffffff; margin: 5px 0 0 0; font-size: 20px; }}
    .dot {{ width: 8px; height: 8px; border-radius: 50%; display: inline-block; margin-right: 5px; }}
</style>

<div class="main-box">
    <!-- CABEÇALHO -->
    <div style="margin-bottom: 30px; border-bottom: 1px solid #1e293b; padding-bottom: 20px;">
        <h1 style="color: #ffffff; font-size: 28px; margin: 0; font-weight:700;">Market Intelligence Dashboard</h1>
        <p style="color: #cbd5e1; font-size: 14px; margin-top: 5px;">📍 Otimização de Compras & Análise Competitiva</p>
    </div>

    <!-- KPI ROW -->
    <div class="kpi-row">
        <div class="kpi-card"><span>MELHOR PREÇO</span><h2>R$ {df_plot.iloc[0]['Custo_Total']:.2f}</h2></div>
        <div class="kpi-card"><span>MÉDIA DO SETOR</span><h2>R$ {df_local['Custo_Total'].mean():.2f}</h2></div>
        <div class="kpi-card"><span>PLATAFORMA DE BUSCA</span><h2>Mercado Livre Brasil</h2></div>
    </div>

    <!-- CONTEÚDO PRINCIPAL -->
    <div style="display: flex; gap: 20px; align-items: flex-start;">
        <!-- LADO ESQUERDO: RANKING -->
        <div style="flex: 2; background: #0f172a; padding: 20px; border-radius: 15px; border: 1px solid #1e293b;">
            <h3 style="color: #ffffff; font-size: 15px; margin-bottom: 10px;">📊 Ranking Top 30 (Clique na barra para ver anúncio)</h3>
            {graph_bar_html}
        </div>

        <!-- LADO DIREITO: BREAKDOWN -->
        <div style="flex: 0.8; display: flex; flex-direction: column; gap: 20px;">
            <div style="background: #0f172a; padding: 20px; border-radius: 15px; border: 1px solid #1e293b; text-align: center;">
                <h3 style="color: #ffffff; font-size: 15px; margin-bottom: 10px;">🚀 Breakdown Logístico</h3>
                {graph_pie_html}
                <!-- LEGENDA ABAIXO DA PIZZA -->
                <div style="display: flex; justify-content: center; gap: 15px; margin-top: 5px;">
                    <div style="font-size: 11px; color: #ffffff;"><span class="dot" style="background:#10B981;"></span>Sim (Full)</div>
                    <div style="font-size: 11px; color: #ffffff;"><span class="dot" style="background:#475569;"></span>Não</div>
                </div>
            </div>

            <div style="background: #10B981; color: #020617; padding: 15px; border-radius: 12px; text-align: center; font-weight: bold; font-size: 13px;">
                ⚡ Entrega Full Priorizada na Análise
            </div>
        </div>
    </div>
</div>

<script>
var plotDiv = document.getElementById('{id_div}');
plotDiv.on('plotly_click', function(data){{
    var url = data.points[0].customdata;
    if(url) window.open(url, '_blank');
}});
</script>
"""

display(HTML(dashboard_html))

In [59]:
import os
from IPython.display import HTML

# --- 1. PREPARAÇÃO DO CONTEÚDO (GRÁFICOS) ---
# Geramos o HTML interno de cada gráfico para injetar no template
bar_html = fig_bar.to_html(full_html=False, include_plotlyjs='cdn', div_id='bar_div')
pie_html = fig_pie.to_html(full_html=False, include_plotlyjs='cdn')
data_atual = pd.Timestamp.now().strftime('%d/%m/%Y %H:%M')

# --- 2. O TEMPLATE COMPLETO (A "ROUPA" DO DASHBOARD) ---
dashboard_completo_html = f"""
<!DOCTYPE html>
<html lang="pt-br">
<head>
    <meta charset="UTF-8">
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;700&display=swap" rel="stylesheet">
    <style>
        body {{ background-color: #020617; margin: 0; padding: 20px; font-family: 'Inter', sans-serif; color: white; display: flex; justify-content: center; }}
        .main-box {{ background: #020617; padding: 35px; border-radius: 20px; border: 1px solid #1e293b; max-width: 1150px; width: 100%; }}
        .kpi-row {{ display: flex; gap: 15px; margin-bottom: 25px; }}
        .kpi-card {{ flex: 1; background: #0f172a; padding: 15px 20px; border-radius: 12px; border: 1px solid #1e293b; }}
        .kpi-card span {{ color: #94a3b8; font-size: 10px; font-weight: bold; text-transform: uppercase; letter-spacing: 1px; }}
        .kpi-card h2 {{ color: #ffffff; margin: 5px 0 0 0; font-size: 20px; }}
        .dot {{ width: 8px; height: 8px; border-radius: 50%; display: inline-block; margin-right: 5px; }}
        .container-flex {{ display: flex; gap: 20px; align-items: flex-start; }}
        .col-left {{ flex: 2; background: #0f172a; padding: 20px; border-radius: 15px; border: 1px solid #1e293b; }}
        .col-right {{ flex: 0.8; display: flex; flex-direction: column; gap: 20px; }}
        .card-inner {{ background: #0f172a; padding: 20px; border-radius: 15px; border: 1px solid #1e293b; text-align: center; }}
        h1, h3 {{ color: white; margin: 0; }}
        a {{ text-decoration: none; }}
    </style>
</head>
<body>
    <div class="main-box">
        <div style="margin-bottom: 30px; border-bottom: 1px solid #1e293b; padding-bottom: 20px;">
            <h1>Market Intelligence Dashboard</h1>
            <p style="color: #cbd5e1; font-size: 14px; margin-top: 5px;">📍 Otimização de Compras & Análise Competitiva</p>
        </div>

        <div class="kpi-row">
            <div class="kpi-card"><span>MELHOR PREÇO</span><h2>R$ {df_plot.iloc[0]['Custo_Total']:.2f}</h2></div>
            <div class="kpi-card"><span>MÉDIA DO SETOR</span><h2>R$ {df_local['Custo_Total'].mean():.2f}</h2></div>
            <div class="kpi-card"><span>PLATAFORMA DE BUSCA</span><h2>Mercado Livre Brasil</h2></div>
        </div>

        <div class="container-flex">
            <div class="col-left">
                <h3 style="font-size: 15px; margin-bottom: 10px;">📊 Ranking Top 30 (Clique na barra para ver anúncio)</h3>
                {bar_html}
            </div>

            <div class="col-right">
                <div class="card-inner">
                    <h3 style="font-size: 15px; margin-bottom: 10px;">🚀 Breakdown Logístico</h3>
                    {pie_html}
                    <div style="display: flex; justify-content: center; gap: 15px; margin-top: 5px;">
                        <div style="font-size: 11px;"><span class="dot" style="background:#10B981;"></span>Sim (Full)</div>
                        <div style="font-size: 11px;"><span class="dot" style="background:#475569;"></span>Não</div>
                    </div>
                </div>
                <div style="background: #10B981; color: #020617; padding: 15px; border-radius: 12px; text-align: center; font-weight: bold; font-size: 13px;">
                    ⚡ Entrega Full Priorizada na Análise
                </div>
            </div>
        </div>
    </div>

    <script>
    // Reabilitando o clique no arquivo exportado
    window.onload = function() {{
        var plotDiv = document.getElementById('bar_div');
        plotDiv.on('plotly_click', function(data){{
            var url = data.points[0].customdata;
            if(url) window.open(url, '_blank');
        }});
    }};
    </script>
</body>
</html>
"""

# --- 3. SALVAMENTO NO DRIVE ---
caminho_final = '/content/drive/MyDrive/RPA_Resultados/DASHBOARD_PROFISSIONAL.html'

with open(caminho_final, 'w', encoding='utf-8') as f:
    f.write(dashboard_completo_html)

print(f"✅ AGORA SIM! Dashboard salvo com sucesso em: {caminho_final}")
print("Baixe este arquivo e abra no seu navegador para ver o resultado real.")

✅ AGORA SIM! Dashboard salvo com sucesso em: /content/drive/MyDrive/RPA_Resultados/DASHBOARD_PROFISSIONAL.html
Baixe este arquivo e abra no seu navegador para ver o resultado real.
